# Figure: Hierarchical KMeans Visualization

Source CLS pool 수집 → 2-step KMeans (M global → K local) → t-SNE 동시 투영.

- Background (gray): Source CLS tokens (post-BasisGAT)
- Foreground (colored): M×K centroids, color = Graph ID (m), edgecolor = black

In [3]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

root_dir = Path('/home/eungyeop/LLM/tabular/ProtoLLM_entropic20251217')
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from models.TabularFLM_S import Model
from dataset.data_dataloaders import prepare_embedding_dataloaders
from utils.util import fix_seed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('>>> device:', device)

>>> device: cuda


In [4]:
# === Checkpoint to visualize =====================================
ckpt_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Medicaldataset+Cardiovascular_Disease_Dataset+Heart_disease_statlog+Erbil_Cardiovascular_Health_Dataset+cardio_SAheart+heart_failure_clinical_records+heart/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-0.5_alpha-0.6_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-None/50/20260413_042538/best_20260413_042538.pt"

ckpt = torch.load(ckpt_path, map_location=device)
args = ckpt['args']
fix_seed(args.random_seed)

model = Model(
    args,
    args.input_dim,
    args.hidden_dim,
    args.output_dim,
    args.dropout_rate,
    args.llm_model,
    experiment_id='figure_kmeans',
    mode='viz',
).to(device)

missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=False)
print('missing:', missing)
print('unexpected:', unexpected)
model.eval()
print('>>> model loaded:', ckpt_path)

>>> [System] Seed 50 FIXED. Deterministic algorithms ENABLED.
missing: []
unexpected: []
>>> model loaded: /storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Medicaldataset+Cardiovascular_Disease_Dataset+Heart_disease_statlog+Erbil_Cardiovascular_Health_Dataset+cardio_SAheart+heart_failure_clinical_records+heart/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-0.5_alpha-0.6_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-None/50/20260413_042538/best_20260413_042538.pt


In [5]:
from torch.utils.data import ConcatDataset, DataLoader

sources = args.source_data if isinstance(args.source_data, (list, tuple)) else [args.source_data]

loaders = {}
for src in sources:
    print(f'>>> loading source: {src}')
    res = prepare_embedding_dataloaders(args, src, is_source=True)
    tr, va, te = res['loaders']

    datasets = [l.dataset for l in [tr, va, te] if l is not None]
    ds = ConcatDataset(datasets)

    loaders[src] = DataLoader(ds, batch_size=32, shuffle=False)
    print(f'    -> total samples: {len(ds)}')

>>> loading source: Medicaldataset
Loading embeddings from: /storage/personal/eungyeop/dataset/embedding/tabular_embeddings_carte/gpt2_mean/embedding_Medicaldataset.pkl
>>> [DataLoader] Created isolated generator with seed 50
[Source] Training data size: 1121
[Source] Validation data size: 198
    -> total samples: 1319
>>> loading source: Cardiovascular_Disease_Dataset
Loading embeddings from: /storage/personal/eungyeop/dataset/embedding/tabular_embeddings_carte/gpt2_mean/embedding_Cardiovascular_Disease_Dataset.pkl
>>> [DataLoader] Created isolated generator with seed 50
[Source] Training data size: 850
[Source] Validation data size: 150
    -> total samples: 1000
>>> loading source: Heart_disease_statlog
Loading embeddings from: /storage/personal/eungyeop/dataset/embedding/tabular_embeddings_carte/gpt2_mean/embedding_Heart_disease_statlog.pkl
>>> [DataLoader] Created isolated generator with seed 50
[Source] Training data size: 229
[Source] Validation data size: 41
    -> total sampl

In [6]:
@torch.no_grad()
def collect_cls_pool(model, loaders, device, max_count=20000):
    """All sources, post-BasisGAT CLS tokens."""
    model.eval()
    all_cls = []
    total = 0

    for src_name, loader in loaders.items():
        for batch in loader:
            batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

            name_embs, val_embs = [], []
            if 'cat_name_embeddings' in batch:
                name_embs.append(batch['cat_name_embeddings'])
            if 'num_name_embeddings' in batch:
                name_embs.append(batch['num_name_embeddings'])
            if 'cat_value_embeddings' in batch:
                val_embs.append(batch['cat_value_embeddings'])
            if 'num_prompt_embeddings' in batch:
                val_embs.append(batch['num_prompt_embeddings'])
            if not name_embs:
                continue

            name = torch.cat(name_embs, dim=1)
            val = torch.cat(val_embs, dim=1)

            x_basis = torch.cat(
                [model.basis_cls.expand(val.size(0), 1, model.input_dim), val], dim=1
            )
            for l in range(model.num_basis_layers):
                norm_x = model.basis_layer_norms[l](x_basis)
                basis_outputs, _ = model.basis_layers[l](name, norm_x)
                x_basis = x_basis + basis_outputs.reshape(
                    x_basis.size(0), x_basis.size(1), model.input_dim
                )

            cls_token = x_basis[:, 0, :].cpu().numpy()
            all_cls.append(cls_token)
            total += cls_token.shape[0]
            if total >= max_count:
                break
        if total >= max_count:
            break

    pool_cls = np.concatenate(all_cls, axis=0)
    print(f'>>> pool_cls: {pool_cls.shape}')
    return pool_cls


pool_cls = collect_cls_pool(model, loaders, device)

>>> pool_cls: (4555, 768)


In [7]:
def hierarchical_kmeans(pool_cls, M, K, seed=42):
    """2-step KMeans: global(M) -> local(K within each group). Returns [M, K, D]."""
    D = pool_cls.shape[1]
    centroids = np.zeros((M, K, D), dtype=np.float32)

    km_global = KMeans(n_clusters=M, n_init=10, random_state=seed).fit(pool_cls)
    for m in range(M):
        group = pool_cls[km_global.labels_ == m]
        if len(group) < K:
            supp_idx = np.random.RandomState(seed).choice(
                len(pool_cls), K - len(group), replace=False
            )
            group = np.concatenate([group, pool_cls[supp_idx]], axis=0)
        km_local = KMeans(n_clusters=K, n_init=10, random_state=seed).fit(group)
        centroids[m] = km_local.cluster_centers_
    return centroids


M = getattr(args, 'n_graphs', 8)
K = getattr(args, 'n_nodes', 8)
print(f'>>> Hierarchical KMeans: M={M}, K={K}')
centroids = hierarchical_kmeans(pool_cls, M=M, K=K, seed=args.random_seed)
print('>>> centroids:', centroids.shape)

>>> Hierarchical KMeans: M=8, K=8
>>> centroids: (8, 8, 768)


In [ ]:
def plot_hierarchical(
    pool_cls,
    centroids,
    num_vis_samples=3000,
    perplexity=30,
    seed=42,
    figsize=(8, 7),
    title='Strategy: HIERARCHICAL',
    save_path=None,
):
    M, K, D = centroids.shape

    rng = np.random.RandomState(seed)
    if pool_cls.shape[0] > num_vis_samples:
        idx = rng.choice(pool_cls.shape[0], num_vis_samples, replace=False)
        vis_cls = pool_cls[idx]
    else:
        vis_cls = pool_cls

    flat = centroids.reshape(-1, D)
    all_data = np.concatenate([vis_cls, flat], axis=0)

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init='pca',
        learning_rate='auto',
        random_state=seed,
    )
    embedded = tsne.fit_transform(all_data)

    split = vis_cls.shape[0]
    emb_cls = embedded[:split]
    emb_cen = embedded[split:]

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(emb_cls[:, 0], emb_cls[:, 1], c='lightgray', alpha=0.2, s=20, label='Source CLS')

    colors = cm.rainbow(np.linspace(0, 1, M))
    for m in range(M):
        sub = emb_cen[m * K:(m + 1) * K]
        ax.scatter(
            sub[:, 0], sub[:, 1],
            color=colors[m], marker='o', s=100,
            edgecolor='black', linewidth=1.5,
            label=f'Graph {m}',
        )

    ax.set_title(title)
    ax.legend(loc='best', ncol=2, fontsize=8)
    plt.tight_layout()

    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'>>> saved: {save_path}')

    plt.show()
    return embedded


_ = plot_hierarchical(pool_cls, centroids, seed=args.random_seed)

## Untrained baseline — early-epoch CLS proxy

같은 architecture / args 로 random-init 모델을 만들어 BasisGAT를 random projection 으로 사용. 학습 초기(epoch 1) 와 정성적으로 동일한 spread 효과.

In [ ]:
fix_seed(args.random_seed)

model_untrained = Model(
    args,
    args.input_dim,
    args.hidden_dim,
    args.output_dim,
    args.dropout_rate,
    args.llm_model,
    experiment_id='figure_kmeans_untrained',
    mode='viz',
).to(device).eval()
print('>>> untrained model ready (no ckpt loaded)')

pool_cls_untrained = collect_cls_pool(model_untrained, loaders, device)

centroids_untrained = hierarchical_kmeans(
    pool_cls_untrained, M=M, K=K, seed=args.random_seed
)
print('>>> centroids_untrained:', centroids_untrained.shape)

_ = plot_hierarchical(
    pool_cls_untrained,
    centroids_untrained,
    seed=args.random_seed,
    title='Strategy: HIERARCHICAL (Untrained / Early-epoch proxy)',
)